# 《数据科学基础》第五、六章：离线学习练习

这些是按教材知识点自编的教学练习（非原文），不是原书代码的逐字转录。先在Obsidian阅读对应章节笔记，再逐格运行。所有材料均为自编HTML或scikit-learn本地自带数据，不访问真实网站。

今天先完成练习一和练习二。后面的回归、聚类与模型比较可分次完成。每做一段，先预测输出，再运行验证，并写一句为什么。

文件保存到E盘当前项目。代码已在本机Python环境执行验证，运行结果保留在单元格中。

In [1]:
from pathlib import Path
import json
import numpy as np
import sklearn

练习目录 = Path(r"E:\Documents\ChatGPT\obsidian\outputs\第五六章练习结果_20261101")
练习目录.mkdir(parents=True, exist_ok=True)
print("结果目录：", 练习目录)
print("scikit-learn：", sklearn.__version__)

结果目录： E:\Documents\ChatGPT\obsidian\outputs\第五六章练习结果_20261101
scikit-learn： 1.9.0


## 练习一：从本地HTML提取三本书

对应第五章§5.2、§5.3。此处从本地字符串开始，练习“解析—提取—保存”；网络请求是爬虫的另一步。目标：理解标签、class属性、文本、href属性、正则提取和缺失字段处理。先找出哪本书缺少价格。

In [2]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re, csv

html = """
<main>
  <article class="book"><h2>Python入门</h2><span class="price">￥39.80</span><a href="/book/1">详情</a></article>
  <article class="book"><h2>数据分析</h2><span class="price">￥58.00</span><a href="/book/2">详情</a></article>
  <article class="book"><h2>机器学习</h2><a href="/book/3">详情</a></article>
</main>
"""
soup = BeautifulSoup(html, "html.parser")
records = []
for card in soup.find_all("article", class_="book"):
    title = card.find("h2").get_text(strip=True)
    price_tag = card.find("span", class_="price")
    price_match = re.search(r"\d+(?:\.\d+)?", price_tag.get_text()) if price_tag else None
    price = float(price_match.group()) if price_match else None
    link = urljoin("https://example.invalid/", card.find("a")["href"])
    records.append({"书名": title, "价格": price, "链接": link})

print(json.dumps(records, ensure_ascii=False, indent=2))
with (练习目录 / "图书信息.csv").open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["书名", "价格", "链接"])
    writer.writeheader()
    writer.writerows(records)
print("已保存3条记录；缺失的价格用None表示，而不是当作0元。")

[
  {
    "书名": "Python入门",
    "价格": 39.8,
    "链接": "https://example.invalid/book/1"
  },
  {
    "书名": "数据分析",
    "价格": 58.0,
    "链接": "https://example.invalid/book/2"
  },
  {
    "书名": "机器学习",
    "价格": null,
    "链接": "https://example.invalid/book/3"
  }
]
已保存3条记录；缺失的价格用None表示，而不是当作0元。


**自己修改后再运行：** 增加第四本书；给“机器学习”补一个价格；把`find_all`改成`find`，观察返回类型差异。最后用自己的话解释：为什么链接取`href`，书名却取文本？

In [3]:
text = "价格：39.80元；评分：9.2分"
print("match：", re.match(r"\d+", text))
print("search：", re.search(r"\d+(?:\.\d+)?", text).group())
print("findall：", re.findall(r"\d+(?:\.\d+)?", text))
tags = "<p>甲</p><p>乙</p>"
print("贪婪：", re.findall(r"<p>(.*)</p>", tags))
print("非贪婪：", re.findall(r"<p>(.*?)</p>", tags))

match： None
search： 39.80
findall： ['39.80', '9.2']
贪婪： ['甲</p><p>乙']
非贪婪： ['甲', '乙']


## 练习二：鸢尾花分类与留出测试集

对应第六章§6.1、§6.2.1、§6.2.2。先认识`X`和`y`：150个样本、每个样本4个特征，目标是三种花的类别。先划分训练集和测试集，再在训练集内部做交叉验证。缩放放入管道，避免预处理先看到验证集。

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X、y：", X.shape, y.shape)
print("训练集、测试集：", X_train.shape, X_test.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsClassifier()),
    {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9]},
    scoring="accuracy", cv=cv, n_jobs=1
)
search.fit(X_train, y_train)
print("只用训练集选出的k：", search.best_params_)
print("训练集内部交叉验证均分：", round(search.best_score_, 4))

X、y： (150, 4) (150,)
训练集、测试集： (120, 4) (30, 4)
只用训练集选出的k： {'kneighborsclassifier__n_neighbors': 3}
训练集内部交叉验证均分： 0.9667


## 练习三：比较模型，但暂不查看测试集

对应§6.2.4—§6.2.6。所有候选模型在同一训练集、同一五折划分上比较。这里的比较是教学演示，不能视为这些算法在所有数据上的排名。模型及超参数确定后，下一格才对留出的测试集做一次评价。

In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

models = {
    "kNN": clone(search.best_estimator_),
    "SVM": make_pipeline(StandardScaler(), SVC(C=1.0, kernel="rbf")),
    "决策树": DecisionTreeClassifier(max_depth=4, random_state=42),
    "随机森林": RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42, n_jobs=1)
}
cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    cv_scores[name] = float(scores.mean())
    print(name, "五折均分：", round(scores.mean(), 4))

# kNN已在相同训练数据上调过参，这不是无偏的算法比较实验；这里只演示API和选择流程。
best_name = max(cv_scores, key=cv_scores.get)
final_model = models[best_name].fit(X_train, y_train)
print("当前演示选中：", best_name)

kNN 五折均分： 0.9667
SVM 五折均分： 0.9667
决策树 五折均分： 0.9417
随机森林 五折均分： 0.95
当前演示选中： kNN


In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pred = final_model.predict(X_test)
print("留出测试集准确率：", round(accuracy_score(y_test, pred), 4))
print("混淆矩阵（行=真实类别，列=预测类别）：")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names, zero_division=0))
新样本 = [[5.1, 3.5, 1.4, 0.2]]
print("新样本预测：", iris.target_names[final_model.predict(新样本)[0]])

留出测试集准确率： 0.9333
混淆矩阵（行=真实类别，列=预测类别）：
[[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

新样本预测： setosa


**自测：** 为什么不直接在150个样本上训练后再报告同一批样本的准确率？`n_neighbors`是超参数，还是拟合后学出的参数？一个新样本为什么仍要写成二维列表？不要为了提高本格测试分数，反复用测试集挑模型。

## 练习四：回归预测的是连续数值

对应§6.2.3。这里有意使用scikit-learn自带的糖尿病**回归**数据，与原书例6.6的768行二分类CSV不是同一份数据。这一格使用`LinearRegression`，评价MSE和R²，不报告“分类准确率”。

In [7]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X_reg, y_reg = load_diabetes(return_X_y=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
reg = LinearRegression().fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)
print("回归数据：", X_reg.shape, y_reg.shape)
print("测试MSE：", round(mean_squared_error(yr_test, yr_pred), 4))
print("测试R²：", round(r2_score(yr_test, yr_pred), 4))
print("10个回归系数：", reg.coef_.shape)

回归数据： (442, 10) (442,)
测试MSE： 2900.1936
测试R²： 0.4526
10个回归系数： (10,)


## 练习五：无标签聚类

对应§6.3。拟合KMeans时仅传入特征，不传入花的类别标签。轮廓系数不需要真实标签；ARI是在拟合结束后，额外用已知类别做对照。簇编号本身不等于花的类别编号。

In [8]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

X_cluster = StandardScaler().fit_transform(iris.data)
km = KMeans(n_clusters=3, n_init=10, random_state=42)
clusters = km.fit_predict(X_cluster)
print("簇中心形状：", km.cluster_centers_.shape)
print("轮廓系数：", round(silhouette_score(X_cluster, clusters), 4))
print("ARI：", round(adjusted_rand_score(iris.target, clusters), 4))
print("注意：iris.target没有传给fit；这里只在事后对照时使用。")

簇中心形状： (3, 4)
轮廓系数： 0.4599
ARI： 0.6201
注意：iris.target没有传给fit；这里只在事后对照时使用。


## 练习六：多层感知机（选做）

对应§6.4。先只理解隐藏层、权重、激活函数与缩放。此处仍仅用训练集做交叉验证，不拿测试集反复试参数。完成后将隐藏层从`(10, 10)`改为`(5,)`，观察均分和运行时间；单次结果不能证明网络越大越好。

In [9]:
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(10, 10), solver="lbfgs", max_iter=1500, random_state=42)
)
mlp_scores = cross_val_score(mlp, X_train, y_train, cv=cv, scoring="accuracy")
print("MLP训练集内部五折均分：", round(mlp_scores.mean(), 4))

MLP训练集内部五折均分： 0.9417


## 学完后交给助手检查

在Obsidian的学习路线页写下：你对爬虫四步骤的解释；一次本地HTML提取的结果；X和y各是什么；训练集、交叉验证、测试集的区别；你对一个混淆矩阵的解释。把不懂的地方标成“我的疑问”，再在聊天中发送。